# Task 4 build_model_v3_full_tta — full-data model and TTA test predictions
`build_model_v3.ipynb` found that averaging five fixed views (`all5`: plain, ±10° rotation, 0.9/1.1 scale) beat the plain prediction on both grouped folds (+0.0031 fold 1, +0.0018 fold 2). This notebook

1. refits the selected geometry procedure on **all labeled Train images** for 7 epochs, exactly as `t4_resnet18_geometry_full_v2` was built (seed 42);
2. predicts Test once per view and writes two per-task files: **plain** (should match the frozen Task 4 column) and **all5 TTA**.

Test images are used only for prediction. The combined Kaggle file is assembled separately in `src/submission/`.

In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
RAW = SRC / "task4" / "data" / "raw"
CACHE = SRC / "task4" / "models" / ".cache"
import numpy as np
import pandas as pd
import joblib
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task4.models.image_training import CLASS_NAMES, fit_full_image_model, save_checkpoint, load_checkpoint
from src.task4.models.tta import VIEWS, view_probabilities, tta_predict
SEED = set_seed(include_torch=True)
SMOKE = False  # True = tiny subset, 1 epoch, nothing saved
DATA_NAME = "eda_near_duplicate_v3"
EPOCHS = 7     # median best epoch of the three geometry folds in build_model_v2 (7, 8, 7)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(SRC / "submission" / "sample_submission.csv", usecols=["id"], dtype=str)
assert test.id.tolist() == sample.id.tolist() and len(test) == 25889
train_paths = [RAW / "train" / f for f in train.file_name]
test_paths = [RAW / "test" / f for f in test.file_name]
labels = train.label.to_numpy(dtype=np.int64)
if SMOKE:
    keep = np.random.default_rng(0).choice(len(train_paths), 256, replace=False)
    train_paths, labels = [train_paths[i] for i in keep], labels[keep]
    test, test_paths = test.head(64), test_paths[:64]
print("Train images:", len(train_paths), "| Test images:", len(test_paths), "| SMOKE =", SMOKE)

Train images: 31546 | Test images: 25889 | SMOKE = False


## 1. Full-data fit
Same call as the cell that produced `t4_resnet18_geometry_full_v2`: pretrained ResNet18, geometry augmentation, AdamW lr 1e-4, class-weighted loss, batch 32, seed 42, 7 epochs. There is no validation set here; the procedure's estimate comes from the grouped folds. About 30 minutes on an RTX 3050.

In [2]:
FULL = "t4_resnet18_geometry_full_v3"
paths = {ext: SRC / "task4" / "save" / f"{FULL}{ext}" for ext in (".pt", ".json", "-evaluation.json")}
if SMOKE:
    model, history = fit_full_image_model(train_paths, labels, variant="geometry", seed=SEED, epochs=1)
elif not all(p.exists() for p in paths.values()):
    assert not any(p.exists() for p in paths.values()), "Inspect incomplete artifact"
    model, history = fit_full_image_model(train_paths, labels, variant="geometry", seed=SEED, epochs=EPOCHS)
    save_model(task=4, name=FULL, model=model, data_name=DATA_NAME, saver=save_checkpoint, extension="pt",
        card={"idea": "Refit of the t4_resnet18_geometry_full_v2 procedure on all labeled Train images (checkpoint kept locally for TTA)",
              "augmentation": "geometry", "epochs": history, "seed": SEED,
              "inference": "image_training.load_checkpoint; tta.view_probabilities"})
    evaluate_model(task=4, name=FULL, score=float(np.mean([0.95246, 0.95528])),
        metric_name="macro_F1_grouped_cv2_procedure", higher_is_better=True,
        card={"note": "Procedure estimate = mean plain macro F1 of build_model_v3 folds 1-2; not measured on this full-data fit"})
else:
    model = load_checkpoint(paths[".pt"])
print(FULL, "ready")

full geometry epoch 1/7: loss=0.4808


full geometry epoch 2/7: loss=0.2472


full geometry epoch 3/7: loss=0.1852


full geometry epoch 4/7: loss=0.1554


full geometry epoch 5/7: loss=0.1298


full geometry epoch 6/7: loss=0.1147


full geometry epoch 7/7: loss=0.1065


t4_resnet18_geometry_full_v3 ready


## 2. Test predictions per view
Probabilities for all five views are computed in one pass and cached, then averaged for `all5`. `plain` uses only the identity view, which reproduces `image_transform('none')`.

In [3]:
cache = CACHE / f"{FULL}_test_view_probs.npz"
if not SMOKE and cache.exists():
    probs = dict(np.load(cache))
else:
    probs = view_probabilities(model, test_paths, list(VIEWS))
    if not SMOKE:
        CACHE.mkdir(parents=True, exist_ok=True); np.savez_compressed(cache, **probs)
plain = tta_predict(probs, ["identity"])
all5 = tta_predict(probs, list(VIEWS))
print("Changed by all5 TTA:", int((plain != all5).sum()), "of", len(plain))
print("Class counts plain:", np.bincount(plain, minlength=5).tolist())
print("Class counts all5: ", np.bincount(all5, minlength=5).tolist())

Changed by all5 TTA: 1360 of 25889
Class counts plain: [4748, 6024, 3572, 6641, 4904]
Class counts all5:  [4633, 6291, 3555, 6534, 4876]


## 3. Per-task files
`src/task4/submission/<name>.csv` with columns `id,task4` (integer class codes as in `train.csv`). The TTA procedure is recorded as its own small model artifact, like in `build_model_v3`.

In [4]:
out_dir = SRC / "task4" / "submission"
TTA = f"{FULL}_tta_all5"
if SMOKE:
    print("SMOKE: nothing written")
else:
    out_dir.mkdir(parents=True, exist_ok=True)
    if not (SRC / "task4" / "save" / f"{TTA}-evaluation.json").exists():
        save_model(task=4, name=TTA, model={"checkpoint": f"task4/save/{FULL}.pt", "views": VIEWS},
            data_name=DATA_NAME, saver=joblib.dump, extension="joblib",
            card={"idea": "all5 TTA on the full-data refit; views fixed from build_model_v3 folds 1-2",
                  "inference": "tta.view_probabilities + tta.tta_predict(all views)"})
        evaluate_model(task=4, name=TTA, score=float(np.mean([0.95561, 0.95705])),
            metric_name="macro_F1_grouped_cv2_procedure", higher_is_better=True,
            card={"note": "Mean all5 TTA macro F1 of build_model_v3 folds 1-2; fold 2 confirmed, fold 1 took part in choosing"})
    for name, pred in ((FULL, plain), (TTA, all5)):
        csv = out_dir / f"{name}.csv"
        frame = pd.DataFrame({"id": test.id, "task4": pred.astype(int)})
        if csv.exists():
            assert pd.read_csv(csv).equals(frame), f"{csv.name} exists with different content"
        else:
            frame.to_csv(csv, index=False)
        print(csv.relative_to(SRC), hashlib.sha256(csv.read_bytes()).hexdigest())

task4\submission\t4_resnet18_geometry_full_v3.csv 0bb2db141ea2abee40bd96a8ff1e9c286b6a551b539928d3ed35468667f1f6df
task4\submission\t4_resnet18_geometry_full_v3_tta_all5.csv 1d5be262dfe574511048ad156030ac8a3e9cf70dc18fdbc7b71452fcaf45c0d9
